# Design Advisory Document pipeline (mid-submission)

**Flow:** Mermaid class diagram → two fine-tuned SLMs (design + code LoRA) → **DAD** → Gemini generates code twice (**baseline** vs **with DAD**) → quick metrics → artifacts saved.

Everything the notebook produces goes into `/content/artifacts/` and is zipped for download at the end.

In [41]:
%%capture
!pip install unsloth
!pip install -q -U google-genai networkx pandas

In [42]:
import os, re, json, glob, time, ast, shutil, subprocess
from google.colab import files

NEEDED = ["design_smell_lora.zip", "code_smell_lora.zip", "design_skill.md", "code_skill.md"]

def find_ci(name):   # case-insensitive, so design_SKILL.md and design_skill.md both work
    return next((f for f in os.listdir(".") if f.lower() == name.lower()), None)

missing = [n for n in NEEDED if not find_ci(n)]
if missing:
    print("Please upload:", missing)
    files.upload()
P = {n: find_ci(n) for n in NEEDED}
assert all(P.values()), f"Still missing: {[n for n in NEEDED if not P[n]]}. Check the file names."
print("Using:", P)

ART = "/content/artifacts"
os.makedirs(ART, exist_ok=True)
print("All inputs found. Artifacts will be saved to", ART)

Using: {'design_smell_lora.zip': 'design_smell_lora.zip', 'code_smell_lora.zip': 'code_smell_lora.zip', 'design_skill.md': 'design_skill.md', 'code_skill.md': 'code_skill.md'}
All inputs found. Artifacts will be saved to /content/artifacts


## 1. Read the SKILL.md files
The skills are used as the single source of truth for: smell meanings (used as DAD explanations), suggested fixes, the DAD table format, and the limitations note. Edit the SKILL.md files and re-run to change the DAD without touching code.

In [43]:
def read(p): return open(p, encoding="utf-8").read()
DESIGN_SKILL, CODE_SKILL = read(P["design_skill.md"]), read(P["code_skill.md"])

def section(md_text, heading):
    m = re.search(rf"^## {re.escape(heading)}\s*\n(.*?)(?=^## |\Z)", md_text, re.S | re.M)
    return m.group(1).strip() if m else ""

def parse_tables(md_text, heading):
    # returns ({smell: (meaning, fix)}, {smells listed under an "Unseen" subsection})
    ref, unseen, sub = {}, set(), ""
    for line in section(md_text, heading).splitlines():
        if line.startswith("###"): sub = line.lower(); continue
        cells = [c.strip().strip("*") for c in line.strip().strip("|").split("|")]
        if len(cells) >= 3 and cells[0] and not set(cells[0]) <= set("-: ") and cells[0] != "Smell":
            ref[cells[0]] = (cells[1], cells[2])
            if "unseen" in sub: unseen.add(cells[0])
    return ref, unseen

DESIGN_REF, DESIGN_UNSEEN = parse_tables(DESIGN_SKILL, "Smells and standard fixes")
CODE_REF,   CODE_UNSEEN   = parse_tables(CODE_SKILL,   "Smells and standard fixes")
assert DESIGN_REF and CODE_REF, "Could not parse the fix tables - check the SKILL.md headings."

# Severity rule for the code model comes from code_SKILL.md ("DAD rows" section)
CODE_SEV = {"Long Parameter List": "medium", "Long Method": "medium",
            "Primitive Obsession": "low", "Magic Number Risk": "low", "Duplicate Code": "low"}

print("Design smells (untrained marked *):", [s + ("*" if s in DESIGN_UNSEEN else "") for s in DESIGN_REF])
print("Code smells   (untrained marked *):", [s + ("*" if s in CODE_UNSEEN else "") for s in CODE_REF])

Design smells (untrained marked *): ['God Class', 'Cyclic Dependency', 'Hub-like Modularization', 'Deep Hierarchy', 'Wide Hierarchy', 'Data Class', 'Middle Man*', 'Multifaceted Abstraction*', 'Insufficient Modularization*']
Code smells   (untrained marked *): ['Long Parameter List', 'Primitive Obsession', 'Long Method', 'Magic Number Risk', 'Duplicate Code*']


## 2. Load both LoRA adapters
Each zip holds a LoRA adapter folder; Unsloth loads the 4-bit Qwen2.5-1.5B base and attaches it. Both fit on one T4.

In [44]:
from unsloth import FastLanguageModel
import torch

def unzip_adapter(zip_path, out):
    os.makedirs(out, exist_ok=True)
    subprocess.run(["unzip", "-o", "-q", zip_path, "-d", out], check=True)
    cfgs = glob.glob(f"{out}/**/adapter_config.json", recursive=True)
    assert cfgs, f"No adapter_config.json found inside {zip_path}"
    return os.path.dirname(cfgs[0])

def load(path, max_len):
    m, t = FastLanguageModel.from_pretrained(path, max_seq_length=max_len, dtype=None, load_in_4bit=True)
    FastLanguageModel.for_inference(m)
    t.padding_side = "left"
    return m, t

DESIGN_LEN, CODE_LEN = 3072, 2048   # same as training
design_path = unzip_adapter(P["design_smell_lora.zip"], "/content/adapters/design")
code_path   = unzip_adapter(P["code_smell_lora.zip"],   "/content/adapters/code")
design_m, design_t = load(design_path, DESIGN_LEN)
code_m,   code_t   = load(code_path,   CODE_LEN)
print("Both models loaded.")

==((====))==  Unsloth 2026.11.2: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).
==((====))==  Unsloth 2026.11.2: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).
Both models loaded.


In [45]:
# Must be identical to the SYSTEM prompts used in training
DESIGN_SYSTEM = (
    "You detect design smells in UML class diagrams written in Mermaid. "
    "Possible smells: God Class, Cyclic Dependency, Hub-like Modularization, "
    "Deep Hierarchy, Wide Hierarchy, Data Class. "
    "Output only a JSON array of objects with keys 'smell', 'class', 'severity' "
    "(high/medium/low). Output [] if there are no smells.")

CODE_SYSTEM = (
    "You predict potential code smells from UML class diagrams written in Mermaid. "
    "Look at the method signatures and attributes. "
    "Possible smells: Long Parameter List, Primitive Obsession, Long Method, Magic Number Risk. "
    "Output only a JSON array of objects with keys 'smell', 'class', 'method' (or 'attribute' if no method), 'reason'. "
    "Output [] if there are no potential code smells.")

TRAINED_D = "God Class, Cyclic Dependency, Hub-like Modularization, Deep Hierarchy, Wide Hierarchy, Data Class"
TRAINED_C = "Long Parameter List, Primitive Obsession, Long Method, Magic Number Risk"
assert TRAINED_D in DESIGN_SYSTEM and TRAINED_C in CODE_SYSTEM
DESIGN_SYSTEM_PROBE = DESIGN_SYSTEM.replace(TRAINED_D, TRAINED_D + ", " + ", ".join(s for s in DESIGN_REF if s in DESIGN_UNSEEN))
CODE_SYSTEM_PROBE   = CODE_SYSTEM.replace(TRAINED_C, TRAINED_C + ", " + ", ".join(s for s in CODE_REF if s in CODE_UNSEEN))

def run_slm(model, tok, system, diagram, max_new=700):
    prompt = tok.apply_chat_template(
        [{"role": "system", "content": system}, {"role": "user", "content": diagram}],
        tokenize=False, add_generation_prompt=True)
    enc = tok(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=False, use_cache=True)
    return tok.decode(out[0, enc.input_ids.shape[1]:], skip_special_tokens=True).strip()

def parse_json(s):
    try: return json.loads(s)
    except Exception:
        m = re.search(r"\[.*\]", s, re.S)
        if m:
            try: return json.loads(m.group(0))
            except Exception: pass
    return None   # invalid

def class_blocks(diagram):
    # class definitions that have members (the code model only needs signatures)
    blocks, cur = [], None
    for line in diagram.splitlines():
        s = line.strip()
        if cur is not None:
            cur.append(line)
            if s == "}": blocks.append("\n".join(cur)); cur = None
        elif s.startswith("class ") and s.endswith("{"):
            cur = [line]
    return blocks

def detect_design(diagram, system=None):
    system = system or DESIGN_SYSTEM
    n_tok = len(design_t(diagram).input_ids)
    budget = DESIGN_LEN - 700 - len(design_t(system).input_ids)
    if n_tok > budget:
        print(f"WARNING: diagram is {n_tok} tokens, budget is ~{budget}. Output may be unreliable - split the diagram.")
    raw = run_slm(design_m, design_t, system, diagram)
    return parse_json(raw), raw

def detect_code(diagram, per_chunk=4, system=None):
    system = system or CODE_SYSTEM
    # long diagrams are split into groups of classes so each prompt stays inside the 2048-token training window
    blocks = class_blocks(diagram)
    findings, raws, invalid = [], [], 0
    for i in range(0, len(blocks), per_chunk):
        chunk = "classDiagram\n" + "\n".join(blocks[i:i + per_chunk])
        raw = run_slm(code_m, code_t, system, chunk)
        raws.append(raw)
        parsed = parse_json(raw)
        if parsed is None: invalid += 1
        else: findings += parsed
    return findings, raws, invalid

## 3. Test diagram
A deliberately smelly, long online-store design (36 classes). Every smell category is planted on purpose, so we can check what the models catch. The `EXPECTED_*` sets below are only for the sanity check in step 4; the models never see them.

In [46]:
DIAGRAM = """classDiagram
  class StoreManager {
    -String storeName
    +createOrder()
    +updateOrder()
    +deleteOrder()
    +findCustomer()
    +loadProduct()
    +saveInvoice()
    +validatePayment()
    +sendNotification()
    +calculateTax()
    +renderReport()
    +exportReport()
    +syncInventory()
    +archiveOrder()
    +scheduleDelivery()
  }
  class CustomerRepository {
    +findCustomer()
    +saveCustomer()
  }
  class ProductRepository {
    +findProduct()
    +saveProduct()
  }
  class InvoiceService {
    +String street
    +String city
    +String country
    +String zipCode
    +double discountPercentage
    +generateReportAndSendEmailAndLogAction()
    +createInvoice()
  }
  class ReportGenerator {
    +renderReport()
  }
  class ShipmentTracker {
    +String street
    +String city
    +String country
    +trackShipment()
    +scheduleDelivery()
  }
  class CheckoutController {
    +checkout()
  }
  class MobileApiGateway {
    +submitOrder()
  }
  class WebApiGateway {
    +submitOrder()
  }
  class AdminConsole {
    +createManualOrder()
  }
  class BatchImporter {
    +importOrders()
  }
  class OrderProcessor {
    +processOrder()
    +cancelOrder()
    +approveOrder()
  }
  class InventoryService {
    +reserveStock()
    +releaseStock()
  }
  class PaymentGateway {
    +String email
    +String phone
    +double taxRate
    +int maxRetryCount
    +processPayment(cardNumber: String, expiry: String, cvv: String, amount: double, currency: String, billingAddress: String, shippingAddress: String)
    +validateAndChargeAndNotifyPayment()
    +refundPayment()
  }
  class AuditLogger {
    +logAction()
  }
  class EmailSender {
    +sendEmail()
  }
  class TaxCalculator {
    +calculateTax()
  }
  class CartService {
    +addItem()
    +removeItem()
    +getTotal()
  }
  class PricingService {
    +priceItem()
  }
  class DiscountService {
    +applyDiscount()
  }
  class BaseEntity
  class Party
  class Person
  class Customer
  class PremiumCustomer
  class VIPCustomer
  class Notification {
    +send()
  }
  class EmailNotification
  class SmsNotification
  class PushNotification
  class WhatsAppNotification
  class SlackNotification
  class VoiceNotification
  class PostalNotification
  class InAppNotification
  class CustomerProfile {
    +String firstName
    +String lastName
    +String street
    +String city
    +String country
    +Date birthDate
    +String preferredLanguage
  }
  StoreManager --> CustomerRepository
  StoreManager --> ProductRepository
  StoreManager --> InvoiceService
  StoreManager --> ReportGenerator
  StoreManager --> ShipmentTracker
  CheckoutController --> OrderProcessor
  MobileApiGateway --> OrderProcessor
  WebApiGateway --> OrderProcessor
  AdminConsole --> OrderProcessor
  BatchImporter --> OrderProcessor
  OrderProcessor --> InventoryService
  OrderProcessor --> PaymentGateway
  OrderProcessor --> AuditLogger
  OrderProcessor --> EmailSender
  OrderProcessor --> TaxCalculator
  CartService --> PricingService
  PricingService --> DiscountService
  DiscountService --> CartService
  Party --|> BaseEntity
  Person --|> Party
  Customer --|> Person
  PremiumCustomer --|> Customer
  VIPCustomer --|> PremiumCustomer
  EmailNotification --|> Notification
  SmsNotification --|> Notification
  PushNotification --|> Notification
  WhatsAppNotification --|> Notification
  SlackNotification --|> Notification
  VoiceNotification --|> Notification
  PostalNotification --|> Notification
  InAppNotification --|> Notification
"""

REQUIREMENTS = """Backend of an online store. Customers browse products, add them to a cart, apply discounts, pay, and receive invoices and notifications (email, SMS, push and other channels). Orders can come from the checkout page, mobile and web APIs, an admin console, and batch imports. The system must reserve inventory, calculate tax, record an audit log, schedule shipments, and let managers export reports. Customers have a profile and a loyalty tier (regular, premium, VIP)."""

# Planted smells, used ONLY to sanity-check the detectors in the next step
EXPECTED_DESIGN = {("God Class","StoreManager"), ("Hub-like Modularization","OrderProcessor"),
    ("Cyclic Dependency","CartService"), ("Cyclic Dependency","PricingService"), ("Cyclic Dependency","DiscountService"),
    ("Deep Hierarchy","VIPCustomer"), ("Wide Hierarchy","Notification"), ("Data Class","CustomerProfile")}
EXPECTED_CODE = {("Long Parameter List","PaymentGateway","processPayment"),
    ("Long Method","PaymentGateway","validateAndChargeAndNotifyPayment"),
    ("Long Method","InvoiceService","generateReportAndSendEmailAndLogAction"),
    ("Primitive Obsession","PaymentGateway","email"), ("Primitive Obsession","PaymentGateway","phone"),
    ("Primitive Obsession","InvoiceService","zipCode"),
    ("Magic Number Risk","PaymentGateway","taxRate"), ("Magic Number Risk","PaymentGateway","maxRetryCount"),
    ("Magic Number Risk","InvoiceService","discountPercentage")}

# Planted for the untrained probe (smell-level check only): Data Clumps = street/city/country in 3 classes,
# Duplicate Code = identical submitOrder() in both API gateways
PLANTED_UNSEEN = {"Data Clumps", "Duplicate Code"}

open(f"{ART}/test_diagram.mmd", "w").write(DIAGRAM)
n_classes = sum(1 for l in DIAGRAM.splitlines() if l.strip().startswith("class "))
print(f"Diagram saved: {n_classes} classes, {len(design_t(DIAGRAM).input_ids)} tokens (design model window: {DESIGN_LEN})")

Diagram saved: 36 classes, 831 tokens (design model window: 3072)


## 4. Run both detectors

In [47]:
t0 = time.time()
design_f, design_raw = detect_design(DIAGRAM)
code_f, code_raw, code_invalid = detect_code(DIAGRAM)
slm_seconds = round(time.time() - t0, 1)

design_invalid = design_f is None
design_f = design_f or []
print(f"SLM inference time: {slm_seconds}s | design JSON invalid: {design_invalid} | code chunks invalid: {code_invalid}")
print(f"Design findings: {len(design_f)} | Code findings: {len(code_f)}")

def loc(f): return f.get("method") or f.get("attribute") or ""
got_d = {(f.get("smell"), f.get("class")) for f in design_f}
got_c = {(f.get("smell"), f.get("class"), loc(f)) for f in code_f}

def report(name, exp, got):
    print(f"\n[{name}] caught {len(exp & got)}/{len(exp)} planted | extra (not planted): {len(got - exp)}")
    for x in sorted(exp - got): print("   MISSED:", x)
    for x in sorted(got - exp): print("   EXTRA :", x)

report("design", EXPECTED_DESIGN, got_d)
report("code", EXPECTED_CODE, got_c)

# ---- Untrained-smell probe: same models, prompt extended with the unseen smells from the skills ----
PROBE_IN_DAD = True    # False = keep untrained findings out of the DAD (cleaner A/B), still reported below
probe_d_all, probe_d_raw = detect_design(DIAGRAM, DESIGN_SYSTEM_PROBE)
probe_c_all, probe_c_raw, probe_c_invalid = detect_code(DIAGRAM, system=CODE_SYSTEM_PROBE)
probe_d_all = probe_d_all or []
probe_design = [f for f in probe_d_all if f.get("smell") in DESIGN_UNSEEN]
probe_code   = [f for f in probe_c_all if f.get("smell") in CODE_UNSEEN]

pd_got = {(f.get("smell"), f.get("class")) for f in probe_d_all}
pc_got = {(f.get("smell"), f.get("class"), loc(f)) for f in probe_c_all}
print("\n=== Untrained-smell probe ===")
print(f"Trained-smell recall with the extended prompt: design {len(EXPECTED_DESIGN & pd_got)}/{len(EXPECTED_DESIGN)} "
      f"(trained prompt: {len(EXPECTED_DESIGN & got_d)}), code {len(EXPECTED_CODE & pc_got)}/{len(EXPECTED_CODE)} "
      f"(trained prompt: {len(EXPECTED_CODE & got_c)})")
print(f"Untrained design findings: {len(probe_design)} | untrained code findings: {len(probe_code)}")
for f in probe_design + probe_code:
    flag = "planted type" if f.get("smell") in PLANTED_UNSEEN else "not planted"
    print(f"   {f.get('smell')} @ {f.get('class')}{('.' + loc(f)) if loc(f) else ''}  [{flag}]")
print("Planted untrained smell types the models flagged:",
      sorted({f.get("smell") for f in probe_design + probe_code} & PLANTED_UNSEEN) or "none")

json.dump({"design": design_f, "code": code_f, "design_raw": design_raw, "code_raw": code_raw,
           "probe_design": probe_design, "probe_code": probe_code,
           "probe_design_raw": probe_d_raw, "probe_code_raw": probe_c_raw},
          open(f"{ART}/slm_findings.json", "w"), indent=2)

Both `max_new_tokens` (=700) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=700) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=700) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=700) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

SLM inference time: 49.5s | design JSON invalid: False | code chunks invalid: 0
Design findings: 5 | Code findings: 17

[design] caught 5/8 planted | extra (not planted): 0
   MISSED: ('Data Class', 'CustomerProfile')
   MISSED: ('Deep Hierarchy', 'VIPCustomer')
   MISSED: ('Hub-like Modularization', 'OrderProcessor')

[code] caught 8/9 planted | extra (not planted): 9
   MISSED: ('Magic Number Risk', 'PaymentGateway', 'maxRetryCount')
   EXTRA : ('Primitive Obsession', 'CustomerProfile', 'city')
   EXTRA : ('Primitive Obsession', 'CustomerProfile', 'country')
   EXTRA : ('Primitive Obsession', 'CustomerProfile', 'firstName')
   EXTRA : ('Primitive Obsession', 'CustomerProfile', 'lastName')
   EXTRA : ('Primitive Obsession', 'CustomerProfile', 'preferredLanguage')
   EXTRA : ('Primitive Obsession', 'CustomerProfile', 'street')
   EXTRA : ('Primitive Obsession', 'InvoiceService', 'city')
   EXTRA : ('Primitive Obsession', 'InvoiceService', 'country')
   EXTRA : ('Primitive Obsession', '

Both `max_new_tokens` (=700) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=700) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=700) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=700) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene


=== Untrained-smell probe ===
Trained-smell recall with the extended prompt: design 5/8 (trained prompt: 5), code 8/9 (trained prompt: 8)
Untrained design findings: 0 | untrained code findings: 1
   Duplicate Code @ PaymentGateway.validateAndChargeAndNotifyPayment  [planted type]
Planted untrained smell types the models flagged: ['Duplicate Code']


## 5. Build the Design Advisory Document
Format comes from the SKILL.md files: `ID | Smell | Location | Severity | Explanation | Suggested fix`, IDs `D1…` for design findings and `C1…` for code findings. Explanations and fixes come from the skills' tables (for code findings, the model's own `reason` is used when present).

In [48]:
SEV_ORDER = {"high": 0, "medium": 1, "low": 2}
esc = lambda s: str(s).replace("|", "\\|").replace("\n", " ")

def build_dad(design_f, code_f, source):
    d_rows, c_rows, seen = [], [], set()
    for f in design_f:
        smell, cls = f.get("smell", "?"), f.get("class", "?")
        if ("D", smell, cls) in seen: continue
        seen.add(("D", smell, cls))
        meaning, fix = DESIGN_REF.get(smell, ("", ""))
        if smell in DESIGN_UNSEEN: meaning = "[untrained probe] " + meaning
        d_rows.append((smell, cls, f.get("severity", "medium"), meaning, fix))
    for f in code_f:
        smell, where = f.get("smell", "?"), f"{f.get('class', '?')}.{loc(f)}".rstrip(".")
        if ("C", smell, where) in seen: continue
        seen.add(("C", smell, where))
        meaning, fix = CODE_REF.get(smell, ("", ""))
        why = f.get("reason") or meaning
        if smell in CODE_UNSEEN: why = "[untrained probe] " + why
        c_rows.append((smell, where, CODE_SEV.get(smell, "low"), why, fix))
    d_rows.sort(key=lambda r: (SEV_ORDER.get(r[2], 3), r[0], r[1]))
    c_rows.sort(key=lambda r: (SEV_ORDER.get(r[2], 3), r[0], r[1]))

    lines = ["# Design Advisory Document", f"Source: {source}",
             f"Findings: {len(d_rows)} design, {len(c_rows)} code (predicted from the diagram; treat as suspected smells)", "",
             "| ID | Smell | Location | Severity | Explanation | Suggested fix |",
             "|----|-------|----------|----------|-------------|---------------|"]
    for i, r in enumerate(d_rows, 1): lines.append(f"| D{i} | " + " | ".join(esc(x) for x in r) + " |")
    for i, r in enumerate(c_rows, 1): lines.append(f"| C{i} | " + " | ".join(esc(x) for x in r) + " |")
    if not d_rows and not c_rows: lines.append("| - | No smells detected | - | - | - | - |")
    lines += ["", "## Limitations", section(DESIGN_SKILL, "Limitations (state these when relevant)"), "",
              section(CODE_SKILL, "Limitations (state these when relevant)")]
    return "\n".join(lines)

DAD = build_dad(design_f + (probe_design if PROBE_IN_DAD else []),
                code_f + (probe_code if PROBE_IN_DAD else []), "test_diagram.mmd")
open(f"{ART}/DAD.md", "w").write(DAD)

from IPython.display import Markdown, display
display(Markdown(DAD))

# Design Advisory Document
Source: test_diagram.mmd
Findings: 5 design, 18 code (predicted from the diagram; treat as suspected smells)

| ID | Smell | Location | Severity | Explanation | Suggested fix |
|----|-------|----------|----------|-------------|---------------|
| D1 | Cyclic Dependency | CartService | medium | A dependency cycle among two or more classes | Introduce an interface or mediator; invert one dependency (DIP); extract shared code |
| D2 | Cyclic Dependency | DiscountService | medium | A dependency cycle among two or more classes | Introduce an interface or mediator; invert one dependency (DIP); extract shared code |
| D3 | Cyclic Dependency | PricingService | medium | A dependency cycle among two or more classes | Introduce an interface or mediator; invert one dependency (DIP); extract shared code |
| D4 | God Class | StoreManager | medium | One class has too many methods (12+ in training) and too many outgoing dependencies (4+) | Split by responsibility (Extract Class); move behaviour next to the data it uses |
| D5 | Wide Hierarchy | Notification | medium | A parent class with 8 or more direct subclasses | Group subclasses under intermediate abstractions, or use Strategy/composition |
| C1 | Long Method | InvoiceService.generateReportAndSendEmailAndLogAction | medium | Method name implies multiple responsibilities. | Split into single-purpose methods; extract helpers |
| C2 | Long Method | PaymentGateway.validateAndChargeAndNotifyPayment | medium | Method name implies multiple responsibilities. | Split into single-purpose methods; extract helpers |
| C3 | Long Parameter List | PaymentGateway.processPayment | medium | Method has more than 5 parameters. | Introduce a parameter object; group related arguments into a class |
| C4 | Duplicate Code | PaymentGateway.validateAndChargeAndNotifyPayment | low | [untrained probe] Method name implies multiple responsibilities. | Extract the shared members into a common base class, mixin, or utility |
| C5 | Magic Number Risk | InvoiceService.discountPercentage | low | Business logic values should be dedicated types, not raw numbers. | Use named constants, configuration, or a dedicated type |
| C6 | Magic Number Risk | PaymentGateway.taxRate | low | Business logic values should be dedicated types, not raw numbers. | Use named constants, configuration, or a dedicated type |
| C7 | Primitive Obsession | CustomerProfile.city | low | Domain concept represented as a raw String. | Create small value classes (`Email`, `Money`, `ZipCode`) with validation |
| C8 | Primitive Obsession | CustomerProfile.country | low | Domain concept represented as a raw String. | Create small value classes (`Email`, `Money`, `ZipCode`) with validation |
| C9 | Primitive Obsession | CustomerProfile.firstName | low | Domain concept represented as a raw String. | Create small value classes (`Email`, `Money`, `ZipCode`) with validation |
| C10 | Primitive Obsession | CustomerProfile.lastName | low | Domain concept represented as a raw String. | Create small value classes (`Email`, `Money`, `ZipCode`) with validation |
| C11 | Primitive Obsession | CustomerProfile.preferredLanguage | low | Domain concept represented as a raw String. | Create small value classes (`Email`, `Money`, `ZipCode`) with validation |
| C12 | Primitive Obsession | CustomerProfile.street | low | Domain concept represented as a raw String. | Create small value classes (`Email`, `Money`, `ZipCode`) with validation |
| C13 | Primitive Obsession | InvoiceService.city | low | Domain concept represented as a raw String. | Create small value classes (`Email`, `Money`, `ZipCode`) with validation |
| C14 | Primitive Obsession | InvoiceService.country | low | Domain concept represented as a raw String. | Create small value classes (`Email`, `Money`, `ZipCode`) with validation |
| C15 | Primitive Obsession | InvoiceService.street | low | Domain concept represented as a raw String. | Create small value classes (`Email`, `Money`, `ZipCode`) with validation |
| C16 | Primitive Obsession | InvoiceService.zipCode | low | Domain concept represented as a raw String. | Create small value classes (`Email`, `Money`, `ZipCode`) with validation |
| C17 | Primitive Obsession | PaymentGateway.email | low | Domain concept represented as a raw String. | Create small value classes (`Email`, `Money`, `ZipCode`) with validation |
| C18 | Primitive Obsession | PaymentGateway.phone | low | Domain concept represented as a raw String. | Create small value classes (`Email`, `Money`, `ZipCode`) with validation |

## Limitations
- The detector was fine-tuned on synthetic diagrams whose labels come from metric thresholds (method counts, dependency counts, hierarchy depth). Treat findings as flags to check, not ground truth.
- Severity is the model's own estimate and is the least reliable output.
- Untrained smells are a generalization probe. Several of them (Feature Envy, Refused Bequest, Divergent Change, Shotgun Surgery) depend on method bodies or change history that a class diagram does not show, so expect low precision and treat them as hints only.
- Large diagrams may exceed the 3072-token context window; split them if so.

- Training labels come from string rules (comma counts, the word "And" in a method name, specific attribute names), so the model has probably learned those rules. Real diagrams with different naming may be missed or misflagged.
- Findings are predictions from signatures only. Confirm them against the generated code with an independent tool.
- Duplicate Code is untrained and needs comparison across classes, which the model sees only within one chunk of the diagram. Treat it as a hint.

## 6. Generate code with Gemini (two conditions)
Same model, same temperature (0.2), same seed, same base prompt. The **only** difference is the DAD block in condition B.

In [49]:
from google import genai
from google.genai import types
from google.colab import userdata

client = genai.Client(api_key=userdata.get("MidSub"))

# Prioritize models confirmed to be available and unblocked in your API catalog
MODEL_CANDIDATES = ["gemini-2.5-flash-lite", "gemini-flash-latest","gemini-3.1-flash-lite"]
MODEL = None
for m in MODEL_CANDIDATES:
    try:
        client.models.generate_content(model=m, contents="Reply with the word OK.")
        MODEL = m; break
    except Exception as e:
        print(f"{m}: {str(e)[:120]}")
if MODEL is None:
    print("None of the candidates worked. Models available to your key:")
    try:
        for m in client.models.list(): print("  ", m.name)
    except Exception as list_err:
        print("  Could not list models:", list_err)
    raise SystemExit("Set MODEL manually.")
print("Using model:", MODEL)

gemini-2.5-flash-lite: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash-lite is no longer available to new
gemini-flash-latest: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand 
Using model: gemini-3.1-flash-lite


In [50]:
try:
    print('Available models for your API key:')
    for m in client.models.list():
        print(f' - {m.name}')
except Exception as e:
    print(f'Error retrieving models: {e}')

Available models for your API key:
 - models/gemini-2.5-flash
 - models/gemini-2.5-pro
 - models/gemini-2.5-flash-preview-tts
 - models/gemini-2.5-pro-preview-tts
 - models/gemma-4-26b-a4b-it
 - models/gemma-4-31b-it
 - models/gemini-flash-latest
 - models/gemini-flash-lite-latest
 - models/gemini-pro-latest
 - models/gemini-2.5-flash-lite
 - models/gemini-2.5-flash-image
 - models/gemini-3-flash-preview
 - models/gemini-3.1-pro-preview
 - models/gemini-3.1-pro-preview-customtools
 - models/gemini-3.1-flash-lite-preview
 - models/gemini-3.1-flash-lite
 - models/gemini-3-pro-image-preview
 - models/gemini-3-pro-image
 - models/nano-banana-pro-preview
 - models/gemini-3.1-flash-image-preview
 - models/gemini-3.1-flash-image
 - models/gemini-3.1-flash-lite-image
 - models/gemini-nano-banana-2.1
 - models/gemini-3.5-flash
 - models/gemini-3.5-flash-lite
 - models/gemini-omni-flash-preview
 - models/gemini-omni-1.1-flash
 - models/gemini-omni-1.2-flash
 - models/gemini-3.5-transcribe
 - mod

In [51]:
BASE_PROMPT = f'''You are a senior Python engineer. Implement the system below as ONE self-contained Python 3.10+ file.

# Requirements
{REQUIREMENTS}

# UML class diagram (Mermaid)
{DIAGRAM}

# Rules
- Implement the design in the diagram: keep the class names, the inheritance relations and the listed operations (an operation may delegate to new helper classes).
- Use in-memory implementations only (no external services, no third-party packages).
- Use type hints and short docstrings.
- End the file with an `if __name__ == "__main__":` demo that runs without errors.
- Output only the code in a single ```python block, with no explanation outside it.'''

DAD_BLOCK = f'''

# Design Advisory Document (DAD)
A Design Advisory Document produced by automated smell detectors is given below. Each row is a suspected design or code smell, with a location and a suggested fix.
- Address every high and medium severity row in your implementation, and low rows where it is cheap to do so.
- Treat rows as suspected smells: if one is clearly a false positive you may ignore it.
- Begin the file with a comment block titled "DAD RESPONSE" listing each DAD ID and one line on how it was addressed or why it was ignored.

{DAD}'''

PROMPT_BASELINE = BASE_PROMPT
PROMPT_DAD = BASE_PROMPT + DAD_BLOCK

# Fallback models in case the chosen model is overloaded (503)
BACKUP_MODELS = [MODEL, "gemini-2.5-flash", "gemini-2.5-pro"]

def gen(prompt, tries=5):
    # Try each available model if 503 issues persist
    for m in BACKUP_MODELS:
        if not m: continue
        print(f"Trying model: {m}...")
        for a in range(tries):
            try:
                t0 = time.time()
                r = client.models.generate_content(model=m, contents=prompt,
                    config=types.GenerateContentConfig(temperature=0.2, seed=42, max_output_tokens=32768))
                if not r.text: raise RuntimeError("empty response")
                u = r.usage_metadata
                return r.text, {"seconds": round(time.time() - t0, 1),
                                "prompt_tokens": getattr(u, "prompt_token_count", None),
                                "output_tokens": getattr(u, "candidates_token_count", None),
                                "thinking_tokens": getattr(u, "thoughts_token_count", None)}
            except Exception as e:
                wait = 5 * (a + 1)
                print(f"  {m} attempt {a+1} failed ({str(e)[:100]}); retrying in {wait}s")
                time.sleep(wait)
    raise RuntimeError("All candidate models failed or returned 503 limits.")

def extract_code(text):
    blocks = re.findall(r"```(?:python)?\s*\n(.*?)```", text, re.S)
    return max(blocks, key=len).strip() if blocks else text.strip()

runs = {}
for name, prompt in [("baseline", PROMPT_BASELINE), ("with_dad", PROMPT_DAD)]:
    print(f"Generating: {name} ...")
    text, usage = gen(prompt)
    code_txt = extract_code(text)
    path = f"{ART}/code_{name}.py"
    open(path, "w").write(code_txt + "\n")
    open(f"{ART}/prompt_{name}.txt", "w").write(prompt)
    runs[name] = {"path": path, "usage": usage}
    print(f"  saved {path}  ({len(code_txt.splitlines())} lines)  usage: {usage}")

Generating: baseline ...
Trying model: gemini-3.1-flash-lite...
  saved /content/artifacts/code_baseline.py  (144 lines)  usage: {'seconds': 3.2, 'prompt_tokens': 1143, 'output_tokens': 1317, 'thinking_tokens': None}
Generating: with_dad ...
Trying model: gemini-3.1-flash-lite...
  saved /content/artifacts/code_with_dad.py  (132 lines)  usage: {'seconds': 2.9, 'prompt_tokens': 2525, 'output_tokens': 1166, 'thinking_tokens': None}


## 7. Quick comparison of the two code files
Cheap `ast`-based proxies (not a replacement for PyExamine/DPy, which your proposal uses for the real evaluation). Lower is better for every row except *classes* and *lines*.

In [52]:
import networkx as nx, pandas as pd
from IPython.display import Markdown, display
import json, subprocess

def metrics(src):
    try: tree = ast.parse(src)
    except SyntaxError as e: return {"syntax_ok": False, "error": str(e)}
    classes = [n for n in ast.walk(tree) if isinstance(n, ast.ClassDef)]
    funcs = [n for n in ast.walk(tree) if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))]
    names = {c.name for c in classes}

    def nparams(f):
        a = f.args
        ps = [p for p in a.posonlyargs + a.args + a.kwonlyargs if p.arg not in ("self", "cls")]
        return len(ps) + bool(a.vararg) + bool(a.kwarg)
    def length(f): return (f.end_lineno or f.lineno) - f.lineno + 1

    magic = 0
    for f in funcs:
        for stmt in f.body:
            for n in ast.walk(stmt):
                if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)) \
                        and not isinstance(n.value, bool) and n.value not in (0, 1, -1):
                    magic += 1

    raw_prims = 0
    PRIM = {"email", "phone", "zip_code", "zipcode", "zip", "currency"}
    for n in ast.walk(tree):
        if isinstance(n, ast.arg) and n.arg.lower() in PRIM and isinstance(n.annotation, ast.Name) and n.annotation.id == "str":
            raw_prims += 1
        if isinstance(n, ast.AnnAssign) and isinstance(n.annotation, ast.Name) and n.annotation.id == "str":
            t = n.target
            nm = t.id if isinstance(t, ast.Name) else getattr(t, "attr", "")
            if nm.lower() in PRIM: raw_prims += 1

    g = nx.DiGraph(); g.add_nodes_from(names)
    for c in classes:
        refs = set()
        for stmt in c.body:
            for n in ast.walk(stmt):
                if isinstance(n, ast.Name) and n.id in names: refs.add(n.id)
                if isinstance(n, ast.Constant) and isinstance(n.value, str) and n.value in names: refs.add(n.value)
        for r in refs - {c.name}: g.add_edge(c.name, r)
    in_cycles = sum(len(s) for s in nx.strongly_connected_components(g) if len(s) > 1)

    parent = {c.name: b.id for c in classes for b in c.bases if isinstance(b, ast.Name) and b.id in names}
    def depth(n):
        d = 0
        while n in parent and d < 50: n = parent[n]; d += 1
        return d

    methods_per_class = [sum(isinstance(x, (ast.FunctionDef, ast.AsyncFunctionDef)) for x in c.body) for c in classes]
    return {"syntax_ok": True,
            "lines": len(src.splitlines()), "classes": len(classes),
            "max_methods_in_a_class": max(methods_per_class, default=0),
            "max_params_in_a_method": max((nparams(f) for f in funcs), default=0),
            "methods_with_>5_params": sum(nparams(f) > 5 for f in funcs),
            "methods_>30_lines": sum(length(f) > 30 for f in funcs),
            "magic_number_literals": magic,
            "raw_str_email/phone/zip/currency": raw_prims,
            "classes_in_dependency_cycles": in_cycles,
            "max_inheritance_depth": max((depth(n) for n in names), default=0)}

rows = {k: metrics(open(v["path"]).read()) for k, v in runs.items()}
df = pd.DataFrame(rows)
# Avoid broken Colab interactive DataFrame rendering by displaying as Markdown
display(Markdown(df.to_markdown()))

# Does each file at least run?
for k, v in runs.items():
    r = subprocess.run(["python", v["path"]], capture_output=True, text=True, timeout=120)
    rows[k]["runs_without_error"] = (r.returncode == 0)
    print(f"{k}: exit code {r.returncode}" + ("" if r.returncode == 0 else f"\n{r.stderr[-400:]}"))

json.dump({k: {**rows[k], "gemini_usage": runs[k]["usage"]} for k in runs},
          open(f"{ART}/metrics.json", "w"), indent=2)
df.to_markdown(f"{ART}/metrics.md")

|                                  |   baseline |   with_dad |
|:---------------------------------|-----------:|-----------:|
| syntax_ok                        |       True |       True |
| lines                            |        144 |        132 |
| classes                          |         30 |         26 |
| max_methods_in_a_class           |          5 |          4 |
| max_params_in_a_method           |          4 |          2 |
| methods_with_>5_params           |          0 |          0 |
| methods_>30_lines                |          0 |          0 |
| magic_number_literals            |          3 |          2 |
| raw_str_email/phone/zip/currency |          0 |          2 |
| classes_in_dependency_cycles     |          0 |          0 |
| max_inheritance_depth            |          5 |          5 |

baseline: exit code 0
with_dad: exit code 0


## 8. Save everything as artifacts

In [53]:
json.dump({"gemini_model": MODEL, "slm_inference_seconds": slm_seconds,
           "design_json_invalid": design_invalid, "code_chunks_invalid": code_invalid,
           "design_findings": len(design_f), "code_findings": len(code_f),
           "planted_design_caught": f"{len(EXPECTED_DESIGN & got_d)}/{len(EXPECTED_DESIGN)}",
           "planted_code_caught": f"{len(EXPECTED_CODE & got_c)}/{len(EXPECTED_CODE)}",
           "probe_in_dad": PROBE_IN_DAD,
           "untrained_design_findings": len(probe_design), "untrained_code_findings": len(probe_code),
           "planted_untrained_types_flagged": sorted({f.get("smell") for f in probe_design + probe_code} & PLANTED_UNSEEN)},
          open(f"{ART}/run_info.json", "w"), indent=2)

shutil.copy(P["design_skill.md"], f"{ART}/design_SKILL.md")
shutil.copy(P["code_skill.md"], f"{ART}/code_SKILL.md")

print("Artifacts:")
for f in sorted(os.listdir(ART)): print("  ", f)

shutil.make_archive("/content/midsub_artifacts", "zip", ART)
files.download("/content/midsub_artifacts.zip")

Artifacts:
   DAD.md
   code_SKILL.md
   code_baseline.py
   code_with_dad.py
   design_SKILL.md
   metrics.json
   metrics.md
   prompt_baseline.txt
   prompt_with_dad.txt
   run_info.json
   slm_findings.json
   test_diagram.mmd


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>